In [ ]:
import sys
import os
import glob
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Add 'src' to path so we can import your custom modules
sys.path.append(os.path.abspath('..'))

from src.parsers import FilenameParser
from src.measurements import (
    LifetimeMeasurement, 
    SpectrumMeasurement, 
    PLMapMeasurement, 
    g2Measurement
)

# 1. SCAN FOR DATA
data_files = sorted(glob.glob("../data_raw/*.csv"))
parsed_measurements = []

for f in data_files:
    meta = FilenameParser.parse(f)
    if meta:
        # Create the appropriate object
        if meta.type == 'Lifetime':
            parsed_measurements.append(LifetimeMeasurement(meta))
        elif meta.type == 'Spectrum':
            parsed_measurements.append(SpectrumMeasurement(meta))
        elif meta.type == 'PL':
            parsed_measurements.append(PLMapMeasurement(meta))
        elif meta.type == 'g2':
            parsed_measurements.append(g2Measurement(meta))

# 2. DEFINE THE INTERACTIVE FUNCTION
def browse_data(index):
    clear_output(wait=True)
    meas = parsed_measurements[index]
    
    # Create the plot
    fig, ax = plt.subplots(figsize=(8, 5))
    meas.plot(ax=ax)
    
    # Display metadata nicely
    print(f"--- Metadata ---")
    print(f"Type:  {meas.meta.type}")
    print(f"File:  {os.path.basename(meas.meta.filepath)}")
    print(f"Coord: X={meas.meta.x}, Y={meas.meta.y}")
    print(f"Power: {meas.meta.power} uW")
    
    plt.tight_layout()
    plt.show()

# 3. CREATE THE UI WITHOUT LINKS
slider = widgets.IntSlider(
    value=0, 
    min=0, 
    max=max(0, len(parsed_measurements)-1), 
    description='File Index:',
    layout=widgets.Layout(width='80%')
)

dropdown = widgets.Dropdown(
    options=[(os.path.basename(m.meta.filepath), i) for i, m in enumerate(parsed_measurements)],
    description='Filename:',
    layout=widgets.Layout(width='80%')
)

# Function to keep slider and dropdown in sync manually
def on_slider_change(change):
    dropdown.value = change['new']

def on_dropdown_change(change):
    slider.value = change['new']

# "Observe" changes instead of "Linking" traits
slider.observe(on_slider_change, names='value')
dropdown.observe(on_dropdown_change, names='value')

# Display
ui = widgets.VBox([dropdown, slider])
out = widgets.interactive_output(browse_data, {'index': slider})

display(ui, out)

Output()